# Modules & Composition

## What's covered

- **Why modules exist** — the reuse problem and the line between "shared code" and "shared mistakes"
- **Root vs child modules** — and the slightly-counterintuitive fact that *every* Terraform config is already a module
- The conventional file structure — `main.tf`, `variables.tf`, `outputs.tf`, `versions.tf`, `README.md`
- The **`module` block** — calling a child, passing inputs, consuming outputs
- **Sources** — local path, public registry, private registry, Git, S3
- **Version constraints** and how to upgrade safely
- The three composition patterns — **wrap**, **layer**, **factory**
- **Anti-patterns** — premature abstraction, the kitchen-sink module, deep nesting
- A worked example — a small `network` module called from a root config


## Why modules

A Terraform configuration that grows past about 500 lines is usually doing too much in one place. Multiple stakeholders care about different parts. Changes to "the network" should be reviewable without scrolling through "the database" code. Splitting along stakeholder boundaries is part of what modules buy you.

The other thing modules buy is **reuse**. The team's standard VPC pattern, the standard IAM role layout, the standard "RDS with KMS encryption and backup" recipe — written once, instantiated everywhere. The first time you write a VPC, you wing it. The third time, you wrap the pattern in a module.

**The trade-off.** Modules add a layer of indirection. To read a module call, you have to read both the call site (which inputs are being passed) and the module itself (what the inputs mean). For very small bits of configuration that aren't actually reused, this indirection costs more than it saves.

The honest rule, repeated from notebook 07 of design-patterns: **the rule of three**. The first time you write a thing, write it. The second time, notice. The third time, extract a module. Modules written for the *first* use case are usually wrong because the second use case wasn't there to constrain them.


## Root vs child modules — every config is a module

A subtle point that confuses newcomers: **every Terraform configuration is a module**. The directory you run `terraform init` in is called the **root module**. Any module called via a `module` block is a **child module** of its caller.

```
   project/
   ├── main.tf            <- the root module
   ├── variables.tf
   ├── outputs.tf
   └── modules/
       └── network/       <- a child module
           ├── main.tf
           ├── variables.tf
           └── outputs.tf
```

When the root module calls `module "network" { source = "./modules/network" }`, the network module is loaded and instantiated. The root passes inputs (which become `var.X` inside the child) and consumes outputs (which the root reads as `module.network.X`).

**What's special about the root module:**

- It's the one you run `terraform` commands against.
- It owns the backend configuration (only root modules can have a `terraform { backend ... }` block).
- It can use `terraform.tfvars` and command-line `-var` flags. Child modules can't.
- It's the only module that has outputs printed by `terraform output`.

Children get their inputs *only* from their callers, never from the environment. This is why you can't pass `TF_VAR_*` env vars into a child module — they're a root-module-only feature.


## Conventional file structure

By convention, a module's files are split by purpose. Terraform doesn't care about filenames — everything in the directory is concatenated logically — but the convention helps readers.

```
   modules/network/
   ├── main.tf         resources (the bulk of the module)
   ├── variables.tf    variable blocks (the inputs)
   ├── outputs.tf      output blocks (the outputs)
   ├── locals.tf       locals (optional; sometimes inline in main.tf)
   ├── versions.tf     terraform { required_version, required_providers }
   ├── README.md       what this module does, inputs, outputs, examples
   └── examples/       runnable example callers
       └── simple/
           └── main.tf
```

A few practical notes:

- **`versions.tf` belongs to *every* module**, including children. It declares the providers the module uses. The root module's providers must satisfy every child's requirements; if the root has `aws = "~> 5.0"` and a child requires `aws = "~> 4.0"`, init fails.
- **Children don't configure providers.** They declare requirements; the root configures them. If a child needs an aliased provider (multi-region setup), the root passes it explicitly via `providers = { aws = aws.eu }` in the `module` block.
- **`examples/` is the readme that actually works.** Module READMEs are notoriously out of date; an examples directory that's tested in CI keeps the docs honest.
- **`terraform-docs`** is the standard tool for auto-generating an inputs/outputs table for the README. Run it in CI.


## Calling a child module

The `module` block instantiates a child:

```hcl
module "network" {
  source  = "./modules/network"
  version = "~> 1.2"            # only for registry sources

  vpc_cidr = "10.0.0.0/16"
  azs      = ["us-east-1a", "us-east-1b", "us-east-1c"]
  tags     = local.common_tags
}

resource "aws_instance" "web" {
  subnet_id = module.network.public_subnet_ids[0]
  # ...
}

output "vpc_id" {
  value = module.network.vpc_id
}
```

Three observations:

- **The block label `"network"`** is the local name. References elsewhere use `module.network.<output_name>`.
- **Inputs are arguments** — `vpc_cidr`, `azs`, `tags` — matching variable names declared inside the module.
- **Outputs are consumed** as `module.network.public_subnet_ids`, `module.network.vpc_id`. The module's `output` blocks define what's exposed.

`count` and `for_each` work on `module` blocks too (Terraform 0.13+). `module.network[0]` and `module.cluster["alpha"]` are valid addresses.


## Sources — where modules come from

A module's `source` argument tells Terraform where to fetch it. Several forms:

| Source | Example | When to use |
|---|---|---|
| **Local path** | `./modules/network` or `../shared/iam` | The module lives in the same repo |
| **Public registry** | `terraform-aws-modules/vpc/aws` | Community modules from registry.terraform.io |
| **Private registry** | `app.terraform.io/myorg/network/aws` | HCP Terraform / private registry |
| **Git** | `git::https://github.com/myorg/tf-modules.git//network?ref=v1.2.0` | Shared modules in a Git repo, no registry |
| **S3** | `s3::https://s3.amazonaws.com/myorg-tf-modules/network.zip` | Private packaged modules in S3 |
| **GitHub shorthand** | `github.com/myorg/tf-modules//network?ref=v1.2.0` | Convenience for GitHub |

**Three notes:**

- **Local paths re-resolve every `init`.** Edits are picked up immediately. No download.
- **Registry and Git sources cache** in `.terraform/modules/`. Bumping a version requires `terraform init -upgrade`.
- **`?ref=v1.2.0`** on Git sources pins to a tag, branch, or commit. **Always pin to a tag**, not to `main`. A teammate cloning your repo at a different time will get a different version of the module otherwise, and your reproducibility evaporates.


## Version constraints — pinning safely

For registry modules, the `version` argument takes a constraint string:

| Constraint | Allows |
|---|---|
| `"1.2.3"` | Exactly 1.2.3 |
| `">= 1.2.0"` | Any version 1.2.0 or newer |
| `"~> 1.2.3"` | Any 1.2.x where x >= 3, but not 1.3.0 (patch only) |
| `"~> 1.2"` | Any 1.x where x >= 2, but not 2.0 (minor or patch) |
| `">= 1.2, < 2.0"` | Any 1.x, 1.2 or later |

**The recommendation.** Pin to a minor version with `~> 1.2` for production. This allows patches automatically but not breaking changes. Use exact pinning (`"1.2.3"`) only when you're tracking a specific build for some reason.

**Upgrading.** Bump the constraint, then `terraform init -upgrade`. *Always* read the module's CHANGELOG before bumping a major version. A `~> 2.0` after `~> 1.2` can mean every resource gets replaced — read the plan very carefully.


## Composition pattern 1 — Wrap

The **wrap** pattern: a thin module that calls one underlying module and pre-fills opinions specific to your organization.

```hcl
# modules/our-vpc/main.tf
module "vpc" {
  source  = "terraform-aws-modules/vpc/aws"
  version = "~> 5.0"

  name = var.name
  cidr = var.cidr

  azs              = var.azs
  private_subnets  = [for i, _ in var.azs : cidrsubnet(var.cidr, 8, i)]
  public_subnets   = [for i, _ in var.azs : cidrsubnet(var.cidr, 8, i + 100)]

  enable_nat_gateway      = true
  single_nat_gateway      = var.env != "prod"   # one NAT in non-prod to save money
  one_nat_gateway_per_az  = var.env == "prod"

  enable_flow_log              = true
  create_flow_log_cloudwatch_log_group = true
  create_flow_log_cloudwatch_iam_role  = true

  tags = merge(var.tags, {
    ManagedBy = "terraform"
    Module    = "our-vpc"
  })
}

output "vpc_id"             { value = module.vpc.vpc_id }
output "private_subnet_ids" { value = module.vpc.private_subnets }
output "public_subnet_ids"  { value = module.vpc.public_subnets }
```

What the wrap buys you: every team across the org gets the same NAT-gateway policy, the same subnet sizing, the same flow-logging defaults, without having to remember the dozens of arguments the underlying module exposes. The wrapper's inputs are minimal — `name`, `cidr`, `azs`, `env`, `tags`. The opinion lives in one place.

**When to wrap.** Whenever you find yourself copy-pasting the same configuration of a community module into three different root configs. The wrapper is the deduplication.


## Composition pattern 2 — Layer

The **layer** pattern: build a stack from the bottom up, each module producing outputs that the next one consumes.

```hcl
# Root module main.tf

module "network" {
  source = "./modules/network"
  cidr   = "10.0.0.0/16"
  azs    = ["us-east-1a", "us-east-1b"]
}

module "iam" {
  source     = "./modules/iam"
  account_id = data.aws_caller_identity.current.account_id
}

module "compute" {
  source = "./modules/compute"

  vpc_id          = module.network.vpc_id
  subnet_ids      = module.network.private_subnet_ids
  instance_role   = module.iam.ec2_role_arn

  instance_count  = 3
}

module "app" {
  source = "./modules/app"

  cluster_id     = module.compute.cluster_id
  target_group   = module.compute.target_group_arn
  image          = "myorg/web:v1.4.2"
}
```

Each layer has a clear input/output contract. Network produces VPC IDs and subnet IDs. IAM produces role ARNs. Compute consumes both and produces cluster + target group. App consumes those and runs the workload.

**The win:** changes are scoped. Swap the compute layer from EC2 to ECS without touching the network or IAM layers — their interfaces don't change. The dependency direction is bottom-to-top; lower layers don't know about higher ones.

**The caveat:** the layering must be acyclic. If `app` outputs feed back into `network`, you've broken the model — usually a sign the boundary is in the wrong place.


## Composition pattern 3 — Factory

The **factory** pattern: one module instantiated N times via `for_each` or `count`, each with different inputs producing parallel infrastructure.

```hcl
locals {
  environments = {
    dev = {
      cidr           = "10.10.0.0/16"
      instance_type  = "t3.micro"
      min_size       = 1
      max_size       = 2
    }
    staging = {
      cidr           = "10.20.0.0/16"
      instance_type  = "t3.small"
      min_size       = 2
      max_size       = 4
    }
    prod = {
      cidr           = "10.30.0.0/16"
      instance_type  = "t3.medium"
      min_size       = 3
      max_size       = 12
    }
  }
}

module "env" {
  for_each = local.environments
  source   = "./modules/environment"

  env_name      = each.key
  cidr          = each.value.cidr
  instance_type = each.value.instance_type
  min_size      = each.value.min_size
  max_size      = each.value.max_size
}

output "env_endpoints" {
  value = { for k, m in module.env : k => m.endpoint }
}
```

Three nearly-identical environments, parameterized by a small map. Adding a fourth environment is one map entry, not a copy-paste.

**The caveat:** the factory pattern makes sense when *the environments really are parallel and isolated*. The moment one environment needs to be wired to another (a shared resource, a peering, an account boundary), the factory pattern breaks down and you go back to explicit modules per environment. This is the boundary between this pattern and the *workspaces* / *separate-state* patterns in notebook 06.


## Designing the module interface

The hardest part of writing a module is the interface. A few principles that hold up:

**Few, well-named inputs.** A module with 50 input variables is hard to use and hard to read. Group related inputs into objects. Provide sensible defaults for everything that isn't load-bearing. The user should be able to instantiate the module with three or four arguments and get a working result.

**Outputs that match the consumer's mental model.** `vpc_id`, `public_subnet_ids`, `private_subnet_ids` — what the next module will need. Don't expose internal resource attributes that no one will use.

**Don't leak implementation details.** A module called `database` should expose `connection_string` and `read_endpoint`, not `aws_rds_cluster.main.id`. The consumer shouldn't have to know whether the implementation is Aurora or single-instance RDS.

**Object inputs for compound options.** Instead of `var.backup_retention_days`, `var.backup_window_start`, `var.backup_window_end` — three variables, easy to set inconsistently — use a single `var.backup = { retention_days = 7, window = "02:00-03:00" }`. The compound object is one logical unit.

**Validate at the boundary.** Use `validation` blocks on variables to enforce invariants — CIDR is well-formed, instance type is in an allowed list, environment is one of `dev / staging / prod`. The errors fire at the module boundary, not three resources deep when something fails to provision.

**Don't accept `count` on the module call as a toggle.** `module "db" { count = var.create_db ? 1 : 0 }` is too easy to break — referencing `module.db[0].endpoint` when `count = 0` errors. Prefer a sub-module pattern or an explicit `var.create_db` argument the module respects internally.


## Anti-patterns

Three module mistakes that surface again and again:

### The premature module

Pulling a five-line resource block into a `module/` directory because it "might be reused." It almost never is. Now reading the root config means jumping between files; the module's interface had to be designed before the second use case existed to inform it; and when the second use case finally arrives, the interface doesn't fit.

**Rule of three** beats this every time. Write the resource block inline. Write it again when you need it again. The third time, *now* you know the shape of the abstraction.

### The kitchen-sink module

One module that creates everything: VPC + IAM + compute + database + monitoring + DNS. Hundreds of variables, hundreds of outputs, no one really understands all of it, and one breaking change in any sub-component blocks everyone.

The fix is **layering** (the pattern above) — split the kitchen-sink into per-domain modules with clear input/output contracts. Each piece can be reasoned about in isolation.

### Deep nesting

Module A calls module B calls module C calls module D. Debugging means traversing four layers of indirection. State paths get long (`module.a.module.b.module.c.module.d.aws_instance.foo`). Refactoring becomes hazardous.

**Two levels of nesting is the practical maximum.** A root module that calls a few opinionated wrappers (one level), each of which may call a community module (two levels), is fine. Three or more levels usually signals that the module hierarchy is doing the wrong thing.


## A worked example — a small network module

The shape of a real module. Inputs, locals, resources, outputs. Three subnets driven by `for_each`, like in notebook 02 — now packaged as a reusable module.

```
modules/network/
├── versions.tf
├── variables.tf
├── locals.tf
├── main.tf
└── outputs.tf
```

`versions.tf`:

```hcl
terraform {
  required_version = ">= 1.6"
  required_providers {
    aws = { source = "hashicorp/aws", version = "~> 5.0" }
  }
}
```

`variables.tf`:

```hcl
variable "name" {
  type        = string
  description = "Prefix used for all resource names."
}

variable "cidr" {
  type        = string
  description = "VPC CIDR block."

  validation {
    condition     = can(cidrhost(var.cidr, 0))
    error_message = "cidr must be a valid CIDR block."
  }
}

variable "azs" {
  type        = list(string)
  description = "List of availability zones to use."
}

variable "tags" {
  type        = map(string)
  default     = {}
  description = "Additional tags to apply to all resources."
}
```

`locals.tf`:

```hcl
locals {
  subnets = {
    for i, az in var.azs : "public_${substr(az, -1, 1)}" => {
      cidr = cidrsubnet(var.cidr, 8, i)
      az   = az
    }
  }

  base_tags = merge(var.tags, {
    Module = "network"
    Name   = var.name
  })
}
```

`main.tf`:

```hcl
resource "aws_vpc" "main" {
  cidr_block           = var.cidr
  enable_dns_hostnames = true
  tags                 = merge(local.base_tags, { Name = var.name })

  lifecycle { prevent_destroy = true }
}

resource "aws_internet_gateway" "main" {
  vpc_id = aws_vpc.main.id
  tags   = local.base_tags
}

resource "aws_subnet" "public" {
  for_each = local.subnets

  vpc_id                  = aws_vpc.main.id
  cidr_block              = each.value.cidr
  availability_zone       = each.value.az
  map_public_ip_on_launch = true
  tags                    = merge(local.base_tags, { Name = "${var.name}-${each.key}" })
}

resource "aws_route_table" "public" {
  vpc_id = aws_vpc.main.id

  route {
    cidr_block = "0.0.0.0/0"
    gateway_id = aws_internet_gateway.main.id
  }

  tags = local.base_tags
}

resource "aws_route_table_association" "public" {
  for_each       = aws_subnet.public
  subnet_id      = each.value.id
  route_table_id = aws_route_table.public.id
}
```

`outputs.tf`:

```hcl
output "vpc_id" {
  value       = aws_vpc.main.id
  description = "ID of the created VPC."
}

output "public_subnet_ids" {
  value       = [for s in aws_subnet.public : s.id]
  description = "IDs of the public subnets, in azs order."
}

output "public_subnets_by_az" {
  value       = { for k, s in aws_subnet.public : s.availability_zone => s.id }
  description = "Map of availability zone to public subnet ID."
}
```

A consumer:

```hcl
module "network" {
  source = "./modules/network"

  name = "prod-main"
  cidr = "10.30.0.0/16"
  azs  = ["us-east-1a", "us-east-1b", "us-east-1c"]
  tags = local.common_tags
}
```

Four inputs in, three outputs out. The next consumer who needs a VPC instantiates this module instead of writing twenty lines of HCL by hand.


## Reading the public registry — what to know

The Terraform Registry at `registry.terraform.io` hosts thousands of community modules. A few worth knowing about:

- **`terraform-aws-modules/vpc/aws`** — the de facto VPC module. Almost certainly the right starting point for AWS networking.
- **`terraform-aws-modules/eks/aws`** — EKS cluster + node groups.
- **`terraform-aws-modules/rds/aws`** — RDS with sensible defaults.
- **`hashicorp/consul/aws`** — Consul deployment.
- **`cloudposse/*`** — a large family of opinionated modules (label/naming, IAM, networking) from Cloud Posse. Strong opinions, well-tested.

**Two warnings about the registry:**

- **Community modules are not curated.** Anyone can publish. A module with 500 stars might still have a security hole or a deeply-nested abstraction that bites you. Read the source before depending on one in production.
- **Major-version bumps are common.** `terraform-aws-modules/vpc/aws` is on v5 as of 2024. Each major bump is a real refactor that may cascade through your code. Pin versions; read CHANGELOGs.

For production work, many teams wrap registry modules behind their own org modules (the **wrap** pattern above) so a registry-side major bump is buffered by a single internal upgrade rather than cascading through every team's root configs.


## Forward

Notebook six turns to **Environments, Workspaces & CI/CD** — how Terraform fits into the actual delivery pipeline. The two competing models for handling dev / staging / production: workspaces or separate state files (and why most teams pick separate state). Directory layout patterns that scale. Continuous-integration pipelines that automate `plan` and gate `apply`. Secret handling so credentials never end up in state. And the tooling ecosystem — Atlantis, Terragrunt, Spacelift, HCP Terraform — that has grown up around Terraform in production teams.
